## 1. What is MCP?

MCP = Model Context Protocol.

Think of it as a standard way for an AI application/agent to interact with external tools and data.

Without MCP:

```
Agent
 ├── custom Python code → Database
 ├── custom Python code → Jira
 ├── custom Python code → API
 └── custom Python code → Files
```

Every integration has its own interface.

With MCP:

```
                    MCP
                     │
        ┌────────────┼────────────┐
        ↓            ↓            ↓
    MCP Server    MCP Server   MCP Server
      DB tools     Jira tools    File tools
```

The agent speaks the MCP protocol, and MCP servers expose standardized capabilities.


## 2. Understand the basic MCP architecture first

For your learning, think of MCP as:

```
                        MCP Client / Host
                            |
                            | MCP protocol
                            |
                        +-----v------+
                        | MCP Server |
                        +-----+------+
                            |
        +-------------------+---------------------------+
        |                   |                           |
      Tool                 Resource                    Prompt
        |                   |                           |
get_weather()        file://incident_report.pdf        incident_analysis_prompt
search_db()          db://customers/123
create_ticket()      config://production

```

The important distinction:

**MCP server**

*"Here are capabilities I expose."*

**MCP client/host**

*"I want to discover and invoke those capabilities."*



Lets take a simple example. 

This is the ```server.py``` in ```mcp_demo``` folder


```
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo Server")


@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


if __name__ == "__main__":
    mcp.run()
    ```



And use this in ```client.py``` :

```
import asyncio
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


server_params = StdioServerParameters(
    command=sys.executable,
    args=["server.py"]
)


async def main():

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            tools = await session.list_tools()

            print("Available tools:")

            for tool in tools.tools:
                print(f"- {tool.name}: {tool.description}")

            result = await session.call_tool(
                "add_numbers",
                {
                    "a": 10,
                    "b": 20
                }
            )
            print("\nResult:")
            print(result)
if __name__ == "__main__":
    asyncio.run(main())
```

**What we're doing now:**

When you run:
```
uv run python client.py
```
the client itself starts server.py:
```
                client.py
                   │
                   │ subprocess
                   ↓
              server.py
                   │
                   │ STDIO
                   ↓
              MCP Client
```
More precisely:
```
client.py
   │
   │ starts a child process
   ▼
python server.py
   │
   ├── stdin  ← MCP requests
   │
   └── stdout → MCP responses
```
So server.py is effectively a local child process, not a remotely hosted service.

That's why you don't need:
```
Docker
Kubernetes
VM
Load Balancer
DNS
HTTP endpoint
```
for this example.

However in production, you CAN host an MCP server

This is where it becomes interesting.

## 3. Deployment patterns in production

### Local MCP

```

Laptop
│
├── Agent
│
└── MCP Server
       │
       └── local database/files
```

Typical use cases:
```
Claude Desktop
VS Code
local development
developer tools
personal assistants
```

### Remote MCP

For production, you might have:
```
                  Internet / VPC
                       │
                       ↓
                 MCP Client
                       │
                       │ network
                       ↓
              ┌─────────────────┐
              │   MCP Server    │
              │                 │
              │   Kubernetes    │
              └────────┬────────┘
                       │
             ┌─────────┼─────────┐
             ↓         ↓         ↓
           DB        APIs      Services
```
Now the MCP server does need to be deployed as a **service**.

For example:
```
Kubernetes
   │
   ├── MCP Server - Customer
   │
   ├── MCP Server - Payments
   │
   └── MCP Server - Incidents
```

## 4. Why would we use a remote MCP server?

Imagine your agent needs access to a production database.

You don't want:
```
Developer Laptop
      │
      ↓
Production Database
```

Instead:

```
Agent
  │
  ↓
MCP Server
  │
  │ authentication
  │ authorization
  │ auditing
  │ rate limiting
  ↓
Production Database
```
The MCP server becomes a controlled boundary around your enterprise systems.

This is especially relevant to your agentic architecture/security preparation.

## 5. Local vs remote deployment


```
			Local MCP	            Remote MCP
Transport	    STDIO	                Network transport
Server	            Local process	        Hosted service
Startup	            Client launches it	    	Already running
Deployment	    Not necessary	        Kubernetes/VM/container etc.
Auth	            Usually simpler	        OAuth/JWT/etc.
Scaling	            Not relevant	        Horizontal scaling
Example	            Local filesystem		Enterprise DB/API

```











## 6. How to work with Multiple MCP Servers


The agent.py reads discovers and read all the available tools from multiple clients. And LLM decides which tool to pick.

```
                       USER
                         │
                         │
                         ▼
                ┌─────────────────┐
                │   Python Host   │
                │    / Agent      │
                └────────┬────────┘
                         │
                  DISCOVER TOOLS
                         │
            ┌────────────┴────────────┐
            │                         │
            ▼                         ▼
     MCP Client #1             MCP Client #2
            │                         │
            ▼                         ▼
     Weather Server            Customer Server
            │                         │
            │                         │
     get_weather()             get_customer()
            │                         │
            └────────────┬────────────┘
                         │
                    all_tools
                         │
                         ▼
                ┌─────────────────┐
                │    OpenAI LLM   │
                │                 │
                │ tools=[         │
                │  get_weather,   │
                │  get_customer   │
                │ ]               │
                └────────┬────────┘
                         │
                    TOOL CALL
                         │
              ┌──────────┴──────────┐
              │                     │
       get_weather             get_customer
              │                     │
              ▼                     ▼
        MCP Client #1         MCP Client #2
              │                     │
              ▼                     ▼
        Weather Server        Customer Server
```


Suppose your agent discovered these tools from MCP servers:


```
openai_tools = [

    {
        "type": "function",
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string"
                }
            },
            "required": ["city"]
        }
    },

    {
        "type": "function",
        "name": "get_customer",
        "description": "Get customer information by customer ID",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string"
                }
            },
            "required": ["customer_id"]
        }
    }
]

```

These came from:

```
Weather MCP
    ↓
get_weather()

Customer MCP
    ↓
get_customer()
```

- Step 1: Send tools to OpenAI

```
from openai import OpenAI

client = OpenAI()

response = client.responses.create(

    model="gpt-5",

    input="What is the weather in Toronto?",

    tools=openai_tools
)
```

What OpenAI actually sees

Conceptually, OpenAI receives something like:

```
SYSTEM:

You have access to these tools:


Tool 1:

Name:
    get_weather

Description:
    Get current weather for a city

Parameters:
    city: string


Tool 2:

Name:
    get_customer

Description:
    Get customer information by customer ID

Parameters:
    customer_id: string



USER:

What is the weather in Toronto?

Now the model reasons internally:

User wants weather information.

Available tools:

1. get_weather
2. get_customer

Weather matches get_weather.

Need argument:

city = Toronto
```
Then OpenAI returns:

```

{
    "type": "function_call",

    "name": "get_weather",

    "arguments": "{\"city\":\"Toronto\"}"
}
```

## 7. Tool calling at scale. What if you have 1000s of tools?


You generally should not send thousands of MCP tool definitions to the LLM on every request. That creates a large context window, higher latency/cost, and worse tool-selection accuracy.

The scalable architecture is usually tool discovery / retrieval before tool execution.

The mental model

Instead of:
```
100 MCP servers
      ↓
5,000 tools
      ↓
send ALL 5,000 tools to LLM
      ↓
LLM chooses one
```

we do

```
                    User request
                         │
                         ▼
                 Intent / Tool Router
                         │
              retrieve relevant tools
                         │
             ┌───────────┴───────────┐
             │                       │
        Weather tools           Customer tools
             │                       │
             └───────────┬───────────┘
                         ▼
                 maybe 5-20 tools
                         │
                         ▼
                       LLM
                         │
                  selects tool
                         │
                         ▼
                    MCP server

```


1. Keep a central tool registry

When your agent starts, MCP clients discover the tools:
```
weather_tools = await weather_session.list_tools()
customer_tools = await customer_session.list_tools()
```
But you don't necessarily pass those tools to OpenAI yet.

Instead, build a registry:
```
tool_registry = {

    "get_weather": {
        "server": "weather",
        "description": "Get weather for a city",
        "session": weather_session
    },

    "get_customer": {
        "server": "customer",
        "description": "Get customer information",
        "session": customer_session
    },

    # thousands more...
}
```
In production, this registry could be backed by a database/search index rather than just Python memory.

2. Index the tool descriptions

Imagine you have:

10,000 tools

You can create embeddings for their descriptions.

For example:
```
get_weather
"Get current weather for a city"

get_customer
"Retrieve customer profile and account information"

create_invoice
"Create an invoice for a customer"

check_fraud
"Check a transaction for potential fraud"

search_medical_document
"Search medical documents for relevant information"
```

Store something like:
```
Tool Registry
─────────────────────────────────────────
tool_name
description
server
input_schema
embedding
permissions
tags
```
Then the user's request:

"What is the weather in Toronto?"

is embedded/searched against the tool registry.

The search might return:

1. get_weather          similarity 0.94
2. get_forecast         similarity 0.89
3. get_location_weather similarity 0.81
4. get_customer         similarity 0.04

Now you only expose:
```
get_weather
get_forecast
get_location_weather
```
to the LLM.

3. Then the LLM does the final selection

Now:
```
relevant_tools = retrieve_tools(user_question)

response = openai_client.responses.create(
    model="gpt-5",
    input=user_question,
    tools=relevant_tools
)
```

Instead of:
```
10,000 tools → LLM

you have:

10,000 tools
     ↓
retrieval
     ↓
10 relevant tools
     ↓
LLM
```

**This is basically RAG for tools.**


But how do you find the right MCP server?

This is another important design question.

You can have a hierarchy.

For example:
```
                    User
                     │
                     ▼
              Intent Router
                     │
          ┌──────────┼──────────┐
          ▼          ▼          ▼
       Finance    Customer    Operations
          │          │          │
          ▼          ▼          ▼
       MCP servers MCP servers MCP servers
          │
          ▼
       Tool retrieval
          │
          ▼
       Top 10 tools
          │
          ▼
          LLM
```
So you don't necessarily search all 10,000 tools.

You can first determine:

Domain = Customer

then search only:

Customer MCP tools
5. There are actually several levels of optimization

A production system might look like:
```
User
 │
 ▼
┌──────────────────────┐
│  Request Router      │
│                      │
│  What domain?        │
│  What capability?    │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│ Tool Registry        │
│                      │
│ semantic search      │
│ metadata filtering   │
│ permissions          │
└──────────┬───────────┘
           │
           ▼
      Top K tools
           │
           ▼
┌──────────────────────┐
│ LLM                  │
│                      │
│ choose tool          │
│ generate arguments   │
└──────────┬───────────┘
           │
           ▼
     MCP Client
           │
           ▼
     MCP Server
           │
           ▼
         Tool

```

### Final architecture recommendation:

We wouldn't expose every MCP tool to the model. We would maintain a centralized capability or tool registry containing metadata such as tool name, description, domain, permissions, server and schema. At request time, We would  use hierarchical routing and semantic retrieval to identify the relevant domain and retrieve a small top-K set of tools. We would  then expose only those tools to the LLM for final tool selection. The selected tool name is mapped back to its MCP session, which executes the call. For very large environments we would also use progressive disclosure, where the model initially sees capabilities and retrieves detailed tool schemas only when needed.
```

                ┌─────────────────┐
                │      User       │
                └────────┬────────┘
                         │
                         ▼
                ┌─────────────────┐
                │ Intent / Router │
                └────────┬────────┘
                         │
                         ▼
                ┌─────────────────┐
                │  Tool Registry  │
                │                 │
                │ metadata        │
                │ embeddings      │
                │ permissions     │
                │ server mapping  │
                └────────┬────────┘
                         │
                   retrieve Top-K
                         │
                         ▼
                ┌─────────────────┐
                │       LLM       │
                │                 │
                │ choose tool     │
                │ generate args   │
                └────────┬────────┘
                         │
                         ▼
                ┌─────────────────┐
                │    MCP Client   │
                └────────┬────────┘
                         │
                         ▼
                ┌─────────────────┐
                │    MCP Server   │
                └────────┬────────┘
                         │
                         ▼
                      Tool

```

One subtle point: if the tool itself changes state or has sensitive permissions, the registry/retrieval layer should also enforce authorization. You don't want semantic retrieval to accidentally make an unauthorized tool available to the LLM.